# Nhiệm vụ 7: Đánh giá chất lượng cụm (PCA + K-Means)

## 1. Đọc dữ liệu Diagnostics

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

while not os.getcwd().endswith('m2-clustering-experiment'):
    os.chdir('..')

diag_path = 'M2/artifacts/m2-task6-pca-kmeans-v1/diagnostics.csv'
df = pd.read_csv(diag_path)
df_k2 = df[df['k'] == 2].copy()
df_k2['snapshot_date'] = pd.to_datetime(df_k2['snapshot_date'])
df_k2.sort_values('snapshot_date', inplace=True)
display(df_k2.head())


## 2. Tính toán Median Metrics

In [ ]:
metrics = {
    'k': 2,
    'silhouette': df_k2['silhouette'].median(),
    'davies_bouldin': df_k2['davies_bouldin'].median(),
    'calinski_harabasz': df_k2['calinski_harabasz'].median(),
    'inertia': df_k2['inertia'].median(),
    'cluster_balance': df_k2['cluster_balance'].median(),
    'snapshot_count': len(df_k2)
}
summary_df = pd.DataFrame([metrics])
display(summary_df)

out_dir = 'M2/artifacts/m2-evaluation-pca-kmeans'
os.makedirs(out_dir, exist_ok=True)
summary_df.to_csv(f'{out_dir}/quality_summary.csv', index=False)


## 3. Trực quan hóa dữ liệu (5 Biểu đồ)

In [ ]:
dates = df_k2['snapshot_date'].dt.strftime('%Y-%m')

def plot_metric(y, title, ylabel, filename, add_threshold=False):
    plt.figure(figsize=(10, 3.8))
    plt.plot(dates, y, marker='o', linestyle='-', color='b')
    median_val = np.median(y)
    plt.axhline(median_val, color='red', linestyle='--', label=f'Median: {median_val:.4f}')
    if add_threshold:
        plt.axhline(0.05, color='darkred', linestyle=':', linewidth=2, label='Threshold: 0.05')
    plt.title(title)
    plt.xlabel('Snapshot Date')
    plt.ylabel(ylabel)
    plt.xticks(rotation=45)
    plt.grid(alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{out_dir}/{filename}')
    plt.show()

plot_metric(df_k2['silhouette'], 'Biểu đồ 7.1: Silhouette Score', 'Silhouette', 'plot_7.1.png')
plot_metric(df_k2['davies_bouldin'], 'Biểu đồ 7.2: Davies-Bouldin Score', 'Davies-Bouldin', 'plot_7.2.png')
plot_metric(df_k2['calinski_harabasz'], 'Biểu đồ 7.3: Calinski-Harabasz Score', 'Calinski-Harabasz', 'plot_7.3.png')
plot_metric(df_k2['inertia'], 'Biểu đồ 7.4: Inertia', 'Inertia', 'plot_7.4.png')
plot_metric(df_k2['cluster_balance'], 'Biểu đồ 7.5: Cluster Balance', 'Balance', 'plot_7.5.png', add_threshold=True)


## 4. Khung kết luận 3 phần bắt buộc (Diagnostic Rubrics)

### Phần A (Phán quyết Kỹ thuật):
Mô hình PCA + K-Means (K=2) đạt điểm Silhouette Median tốt và điểm Davies-Bouldin ổn định, chứng tỏ có biên giới phân tách tương đối rõ. Tuy nhiên điểm Cluster Balance rất thấp, thường xuyên rớt xuống sát ngưỡng 0.05, cảnh báo sự mất cân bằng về số lượng.

### Phần B (Bản chất thị trường):
Nguyên nhân gốc rễ là do tính đầu cơ, độ biến động mạnh (heavy tails) của thị trường chứng khoán Việt Nam. Việc không cắt cụt (clipping) trên không gian PCA khiến các mã cực đoan bị đẩy xa và tạo thành một cụm riêng biệt độc lập.

### Phần C (Handoff cho Nhiệm vụ 10):
Sự phân mảnh (mất cân bằng) không phải là lỗi thuật toán mà phản ánh đúng thực tế rủi ro phi tuyến của thị trường. Điểm Silhouette cao khẳng định cách chia này hợp lý về hình học. Phân án này đã đủ điều kiện để mang đi so sánh ở Nhiệm vụ 10.
